# Implementation of PPO for Generative AI:

## Step 1: Import Libraries 
Importing libraries like Numpy, Transformers and Pytorch modules.

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("hf_token")

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from torch.utils.data import Dataset, DataLoader
import numpy as np
from collections import deque
import random

2026-01-16 10:16:38.556047: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768558598.775749      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768558598.843146      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768558599.404512      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768558599.404550      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768558599.404553      24 computation_placer.cc:177] computation placer alr

## Step 2: Environment Setup 
-Setup device and model: Using GPU if available, loading GPT-2 model and tokenizer. 

-Prepare tokenizer and move model: Setting padding token and moving model to device i.e. GPU or CPU. 

-Optimizer: Using Adam optimizer for training.

In [3]:
class MiniPPO:
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = GPT2LMHeadModel.from_pretrained('gpt2')
        self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-5)
        self.epsilon = 0.2

## Step 3: Training

1. Prepare input and generate text:

Encoding the prompt into tokens and send to device.

Letting GPT-2 generate continuation up to 30 tokens.

Decoding generated tokens to readable text.

2. Compute probabilities:

Feeding generated sequence back to GPT-2 to get logits.

Converting logits to log probabilities of each token.

3. Select log probs of generated tokens: Picking only the log probabilities for the generated words.

4. Compute reward:

Base reward = text length / 25 (max 1).

Bonus +0.5 if text contains “good” or “great”.

5. Compute loss and update model:

Loss = negative log-prob * reward which encourages high reward text.

Backpropagating loss and step optimizer.

Returning generated text and reward.

In [4]:
class MiniPPO:
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = GPT2LMHeadModel.from_pretrained('gpt2').to(self.device)
        self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-5)
        self.epsilon = 0.2 # Marge de clipping PPO

    def train_step(self, prompt):
        self.model.eval() # On génère d'abord sans calcul de gradient
        inputs = self.tokenizer.encode(prompt, return_tensors='pt').to(self.device)

        # 1. GÉNÉRATION ET ANCIENNES LOG_PROBS
        with torch.no_grad():
            outputs = self.model.generate(
                inputs, max_length=inputs.shape[1] + 20, do_sample=True,
                return_dict_in_generate=True, output_scores=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
            gen_sequence = outputs.sequences[0]
            gen_tokens = gen_sequence[inputs.shape[1]:]
            
            # Calcul des probabilités de la politique "actuelle" (qui deviendra l'ancienne)
            old_outputs = self.model(gen_sequence.unsqueeze(0))
            old_logits = old_outputs.logits[:, inputs.shape[1]-1:-1, :]
            old_log_probs = torch.log_softmax(old_logits, dim=-1)
            old_action_log_probs = old_log_probs[0].gather(1, gen_tokens.unsqueeze(-1)).squeeze(-1).detach()

        # 2. CALCUL DE LA RÉCOMPENSE
        text = self.tokenizer.decode(gen_tokens, skip_special_tokens=True)
        reward = min(len(text.split()) / 20, 1.0)
        if any(w in text.lower() for w in ['good', 'great', 'awesome']):
            reward += 0.8
        reward_tensor = torch.tensor([reward], device=self.device)

        # 3. OPTIMISATION (LE CŒUR DE PPO)
        self.model.train()
        # On recalcule les log_probs pour autoriser le gradient
        current_outputs = self.model(gen_sequence.unsqueeze(0))
        current_logits = current_outputs.logits[:, inputs.shape[1]-1:-1, :]
        current_log_probs = torch.log_softmax(current_logits, dim=-1)
        current_action_log_probs = current_log_probs[0].gather(1, gen_tokens.unsqueeze(-1)).squeeze(-1)

        # Ratio entre nouvelle et ancienne politique
        ratio = torch.exp(current_action_log_probs - old_action_log_probs)

        # Formule de perte PPO (Clipped Objective)
        surr1 = ratio * reward_tensor
        surr2 = torch.clamp(ratio, 1.0 - self.epsilon, 1.0 + self.epsilon) * reward_tensor
        loss = -torch.min(surr1, surr2).mean()

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return text, reward


## Step 4: Track Rewards

In [5]:
ppo = MiniPPO()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
history = []
prompts = [
    "The weather today is",
    "Learning AI is",
    "The movie was",
    "This new recipe is"
]

print("Début de l'entraînement...\n")

for i in range(1, 51):
    # On choisit un prompt au hasard parmi notre liste
    prompt = prompts[i % len(prompts)]
    
    # Exécution d'une étape d'entraînement
    text, reward = ppo.train_step(prompt)
    history.append(reward)
    
    # Affichage de la progression tous les 10 pas
    if i % 10 == 0:
        avg_reward = sum(history[-10:]) / 10
        print(f"Étape {i} | Récompense moyenne (10 derniers) : {avg_reward:.2f}")
        print(f"Dernier texte généré : '{text}'")
        print("-" * 30)

print("\nEntraînement terminé !")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Début de l'entraînement...

Étape 10 | Récompense moyenne (10 derniers) : 1.05
Dernier texte généré : ' made in North Carolina, and this was a pretty successful film - well, it was not a big'
------------------------------
Étape 20 | Récompense moyenne (10 derniers) : 1.09
Dernier texte généré : ' perfect and with the high winds over the state, it feels like sunny days on the horizon. We'
------------------------------
Étape 30 | Récompense moyenne (10 derniers) : 1.02
Dernier texte généré : ' written by Christopher L. Johnson - also known as The Black Knight - and is directed by Kevin Mac'
------------------------------
Étape 40 | Récompense moyenne (10 derniers) : 1.11
Dernier texte généré : ' very good but still very sunny. I would have expected to experience some cloud cover.

The'
------------------------------
Étape 50 | Récompense moyenne (10 derniers) : 0.94
Dernier texte généré : ' written by Rob Bowman, writing for the website Cineplex.

On May 9, "'
-------------------------